# MMS ASR on Fon: quick check (Kaggle)

Checks Meta's `facebook/mms-1b-all` with its Fon adapter on ~100 real JML clips. It is meant as the
**second ASR for scoring TTS voices**: unlike kanbior-asr and finanfa-asr-v6, it was not trained on JML
(MMS used Bible readings), so it does not favour voices trained on JML. It is CC-BY-NC-4.0: evaluation only.

Questions answered here:
1. Does the Fon adapter load and transcribe?
2. Can it write Fon tones? (its alphabet is printed and compared with the JML text)
3. Its WER/CER on real recordings: the floor to subtract when it scores TTS audio.
4. Speed on a T4.

**No heavy download:** the clips are streamed from the start of one JML archive (~100 MB read, not 2 GB).

**Kaggle setup:** *Settings → Accelerator*: **GPU T4** (also runs on CPU, slower). *Internet*: **on**. No secrets needed.

In [ ]:
MODEL_ID = "facebook/mms-1b-all"
LANG = "fon"
JML_REPO = "JaimeMaLangue/JML-fon-speech-v1"
ARCHIVE = "audio_transcribed_000.tar"
N_CLIPS = 100            # clips to score
MAX_PER_SPEAKER = 3      # spread the clips over many speakers
CLIP_DIR = "/tmp/jml-mms"

In [ ]:
!pip install -q "transformers>=5.24,<6" "jiwer>=4.0"

In [ ]:
import csv
import re
import tarfile
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from huggingface_hub import HfFileSystem, hf_hub_download
from transformers import AutoProcessor, Wav2Vec2ForCTC

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", device)

## Text normalization

Same rules as `kanbior-mcore/fon_text.py` (copied so this notebook stands alone), so scores are comparable
with kanbior-asr: lowercase, no punctuation, with and without tone marks.

In [ ]:
CHAR_MAP = {"‚": ",", "‘": "'", "’": "'", "ε": "ɛ", "Ε": "Ɛ", "Ð": "Ɖ",
            "“": "", "”": "", '"': "", "«": "", "»": "", "*": ""}
_CHAR_RE = re.compile("|".join(re.escape(k) for k in CHAR_MAP))
_SPACE_RE = re.compile(r"\s+")
_SPACE_BEFORE_PUNCT_RE = re.compile(r"\s+([.,;:!?])")

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = _CHAR_RE.sub(lambda m: CHAR_MAP[m.group(0)], text)
    text = _SPACE_BEFORE_PUNCT_RE.sub(r"\1", _SPACE_RE.sub(" ", text))
    return unicodedata.normalize("NFC", text).strip()

def normalize_for_wer(text, keep_tones=True):
    text = clean_text(text).lower()
    if not keep_tones:
        text = "".join(ch for ch in unicodedata.normalize("NFD", text) if unicodedata.category(ch) != "Mn")
    text = "".join(ch if (ch.isalnum() or ch in " '" or unicodedata.category(ch) == "Mn") else " " for ch in text)
    return unicodedata.normalize("NFC", _SPACE_RE.sub(" ", text)).strip()

def score(refs, preds):
    out = {}
    for suffix, keep in (("", True), ("_no_tones", False)):
        r = [normalize_for_wer(t, keep) for t in refs]
        p = [normalize_for_wer(t, keep) for t in preds]
        out["wer" + suffix] = 100 * jiwer.wer(r, p)
        out["cer" + suffix] = 100 * jiwer.cer(r, p)
    return out

## Clips: streamed from one JML archive

Reads the tar as a stream and stops once `N_CLIPS` usable clips are saved (no digits in the text, at most
`MAX_PER_SPEAKER` per speaker). These speakers may be in kanbior-asr's training set: fine for MMS, which never
saw JML, but kanbior-asr scores on these clips would be optimistic.

In [ ]:
csv_path = hf_hub_download(JML_REPO, "transcribed.csv", repo_type="dataset")
with open(csv_path, newline="", encoding="utf-8") as f:
    meta = {r["audio_path"]: r for r in csv.DictReader(f)}

clip_dir = Path(CLIP_DIR)
clip_dir.mkdir(parents=True, exist_ok=True)
per_speaker = Counter()
clips = []
t = time.time()
with HfFileSystem().open(f"datasets/{JML_REPO}/{ARCHIVE}", "rb", block_size=4 * 2**20) as f:
    with tarfile.open(fileobj=f, mode="r|") as tar:
        for m in tar:
            row = meta.get(m.name)
            if not (m.isfile() and row) or re.search(r"\d", row["transcription"]):
                continue
            if per_speaker[row["speaker_id"]] >= MAX_PER_SPEAKER:
                continue
            path = clip_dir / Path(m.name).name
            path.write_bytes(tar.extractfile(m).read())
            per_speaker[row["speaker_id"]] += 1
            clips.append({"id": row["id"], "path": path, "speaker_id": row["speaker_id"],
                          "text": clean_text(row["transcription"]), "duration": float(row["duration"])})
            if len(clips) == N_CLIPS:
                break

hours = sum(c["duration"] for c in clips) / 3600
print(f"{len(clips)} clips, {hours * 60:.1f} min of audio, {len(per_speaker)} speakers ({time.time() - t:.0f} s)")

## Can MMS write Fon tones?

A CTC model can only output characters in its alphabet. Any tone-marked letter of the JML text that MMS
cannot write is a guaranteed error, whatever it hears. This measures that floor.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, target_lang=LANG)
vocab = {k for k in processor.tokenizer.get_vocab() if len(k) == 1}
print("MMS Fon alphabet:", "".join(sorted(vocab)))

def writable(ch):   # as one token, or as base letter + combining mark(s)
    return ch in vocab or all(p in vocab for p in unicodedata.normalize("NFD", ch))

ref_chars = Counter(ch for c in clips for ch in normalize_for_wer(c["text"]).replace(" ", ""))
missing = {ch: n for ch, n in ref_chars.items() if not writable(ch)}
share = 100 * sum(missing.values()) / sum(ref_chars.values())
print(f"\nCharacters in the references MMS cannot write: {share:.1f}% of all characters")
for ch, n in sorted(missing.items(), key=lambda x: -x[1]):
    print(f"  {ch}  ({' + '.join(f'U+{ord(p):04X}' for p in unicodedata.normalize('NFD', ch))})  x{n}")

## Transcribe

In [ ]:
dtype = torch.float16 if device == "cuda" else torch.float32
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID, target_lang=LANG, ignore_mismatched_sizes=True,
                                       dtype=dtype).to(device).eval()
print(f"{MODEL_ID} [{LANG}]: {model.num_parameters() / 1e6:.0f}M parameters")

preds = []
t = time.time()
for c in clips:
    audio, sr = sf.read(c["path"], dtype="float32")
    assert sr == 16000, f"{c['id']}: expected 16 kHz, got {sr}"
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(inputs.input_values.to(dtype)).logits
    preds.append(processor.decode(logits[0].argmax(dim=-1)))
elapsed = time.time() - t
audio_s = sum(c["duration"] for c in clips)
print(f"{len(clips)} clips in {elapsed:.0f} s: real-time factor {elapsed / audio_s:.3f} on {device}")

## Scores on real recordings

This is the floor: when MMS later scores TTS voices, compare each voice's WER with these numbers.

In [ ]:
refs = [c["text"] for c in clips]
m = score(refs, preds)
print(f"MMS {LANG} on {len(clips)} JML clips:")
print(f"  WER {m['wer']:.1f}%   no tones {m['wer_no_tones']:.1f}%")
print(f"  CER {m['cer']:.1f}%   no tones {m['cer_no_tones']:.1f}%")
print("\nFor reference, kanbior-asr on the JML test set: WER 20.5 / 15.5 no tones, CER 8.5 / 6.6")

pred_marks = sum(unicodedata.category(ch) == "Mn" for p in preds for ch in unicodedata.normalize("NFD", p))
ref_marks = sum(unicodedata.category(ch) == "Mn" for r in refs for ch in unicodedata.normalize("NFD", normalize_for_wer(r)))
print(f"\nTone marks written: {pred_marks} in the predictions vs {ref_marks} in the references")

In [ ]:
from IPython.display import Audio, display

for c, p in list(zip(clips, preds))[:8]:
    print(f"{c['id']} ({c['speaker_id']})")
    display(Audio(str(c["path"])))
    print("  ref :", normalize_for_wer(c["text"]))
    print("  mms :", normalize_for_wer(p), "\n")

## Reading the result

- **Usable as a second scorer** if WER without tones is in a useful range (well below ~60%): then
  differences between TTS voices will show above its own error.
- **Tones:** if it writes few tone marks, or the share of unwritable characters is high, use only its
  *no tones* scores; tones are then judged by kanbior-asr and by native listeners.
- These clips are read speech of everyday sentences, not Bible text: a gap here is partly domain shift,
  which also applies to TTS audio of the same sentences.